<a href="https://colab.research.google.com/github/v4life/Alzheimers/blob/main/Alzheimers_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
API_KEY = "" # paste openAI API key here

## Step 0: Import everything

In [ ]:
#@title Loading in necessary packages and OpenAI API helper functions

!pip install -q openai
!pip install -q langchain-openai
!pip install -q llama-index
!pip install -q llama-index-readers-web
!pip install -q llama-index-embeddings-huggingface
!pip install -q python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.6/123.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.1/565.1 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.9/164.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 11.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cpu requires setuptools<82, but you have setuptools 84.0.0 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from google.colab import data_table
import nltk
import requests
import openai
from openai import OpenAI
import pandas as pd
import json
import numpy as np
import os
import tqdm.notebook as tq
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
def get_openai_response(model_name, prompt, temperature=0.0, max_tokens=None, frequency_penalty=0, top_p=1):
    assert API_KEY

    client = OpenAI(api_key=API_KEY)

    if model_name in ["davinci-002", "gpt-3.5-turbo-instruct", "babbage-002"]:
        response = client.completions.create(
            model=model_name,
            prompt=prompt,
            temperature=temperature
        )


        return response.choices[0].text

    else:
        messages = [
            {
                "role": "system",
                "content": "You are a helpful assistant speaking to an Alzheimer's patient"
            },
            {
                "role": "user",
                "content": prompt
            }
        ]

        response = client.chat.completions.create(
            messages=messages,
            model=model_name,
            temperature=temperature,
            max_tokens=max_tokens,
            frequency_penalty=frequency_penalty,
            top_p=top_p
        )

        return response.choices[0].message.content

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Here's how you generate an embedding for a piece of text
def embedding_fn(text):
    embeddings = embed_model.get_text_embedding(text)
    embeddings = np.expand_dims(np.array(embeddings), axis=0)
    return embeddings

In [ ]:
def distance_fn(prompt, chunk):
    # 1. Create embeddings for prompt and chunk
    prompt_embedding = embedding_fn(prompt)
    chunk_embedding = embedding_fn(chunk)

    # 2. Use the embeddings to compute the cosine similarity using the cosine_similarity function
    cosine = cosine_similarity(prompt_embedding, chunk_embedding)

    return cosine.item()

In [ ]:
def retrieve_top_k_chunks(prompt, chunks, k=1):

  data = {
    'Chunk': [],
    'Cosine Similarity': []
  }

  for i in tq.tqdm(range(len(chunks))):
    chunk = chunks[i]
    #print(f"Computing cosine sim for chunk [{i + 1}/{len(chunks)}]")
    # 1. Compute the cosine sim between the prompt and the chunk's text. Recall that you access the actual string via chunk.text
    try:
      chunk = chunk.text
    except AttributeError:
      chunk = str(chunk)
    cosine = distance_fn(prompt, chunk)

    # 2. Append the chunk's text and its cosine sim with respect to the prompt in the dictionary
    data['Chunk'].append(chunk)
    data['Cosine Similarity'].append(cosine)

  # Create a dataframe from the data object
  df = pd.DataFrame(data).sort_values('Cosine Similarity', ascending=False)
  display(df)
  return df['Chunk'][df.index[0:k]].values

## Step 1: Get your corpus

In [ ]:
benchmark = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vRxPaKPHtko24fSkTbgf7W2yQi59VrEe2vGHGlAKCFhnsH8tytvTv29Lp-0Q7AoXxiQCpX3Nc-EkmVS/pub?gid=0&single=true&output=csv")

## Step 3: Inject the relevant context into your original prompt to correctly answer the question



In [ ]:
model = "gpt-3.5-turbo"

In [ ]:
def insert_newlines(string, every=80):
  #return string
  return '\n'.join(string[i:i+every] for i in range(0, len(string), every))

In [ ]:
prompt_parts = {}
prompt_parts['priorities'] = "When interacting with users who may have Alzheimer's disease, prioritize clarity, reassurance, and simplicity."
prompt_parts['directness'] = "Respond directly to the user's immediate question using short, concrete sentences and only the information needed to answer it."
prompt_parts['one_idea'] = "Present one idea at a time, avoiding unnecessary details, multiple options, or unrelated context."
prompt_parts['language'] = "Use clear, familiar vocabulary and explicit names instead of vague references."
prompt_parts['memory'] = "Provide memory support by supplying needed information rather than testing the user's recall."
prompt_parts['emotion'] = "Validate emotions before offering reassurance or guidance."
prompt_parts['guidance'] = "When helping with a task, provide only the next step. When responding to confusion or word-finding difficulty, offer a simple cue rather than a lengthy explanation."
prompt_parts['tone'] = "Maintain a warm, calm, and encouraging tone throughout every interaction."

In [ ]:
full_prompt = " ".join([prompt_parts[x] for x in prompt_parts.keys()])
ablation_priorities = " ".join([prompt_parts[x] for x in prompt_parts.keys() if x != "priorities"])
ablation_directness = " ".join([prompt_parts[x] for x in prompt_parts.keys() if x != "directness"])
ablation_one_idea = " ".join([prompt_parts[x] for x in prompt_parts.keys() if x != "one_idea"])
ablation_language = " ".join([prompt_parts[x] for x in prompt_parts.keys() if x != "language"])
ablation_memory = " ".join([prompt_parts[x] for x in prompt_parts.keys() if x != "memory"])
ablation_emotion = " ".join([prompt_parts[x] for x in prompt_parts.keys() if x != "emotion"])
ablation_guidance = " ".join([prompt_parts[x] for x in prompt_parts.keys() if x != "guidance"])
ablation_tone = " ".join([prompt_parts[x] for x in prompt_parts.keys() if x != "tone"])

In [ ]:
system_prompts = {
    "full": full_prompt,
    "ablation_priorities": ablation_priorities,
    "ablation_directness": ablation_directness,
    "ablation_one_idea": ablation_one_idea,
    "ablation_language": ablation_language,
    "ablation_memory": ablation_memory,
    "ablation_emotion": ablation_emotion,
    "ablation_guidance": ablation_guidance,
    "ablation_tone": ablation_tone,
}

In [ ]:
def generate_prompt(profile, instruction, query):
  return """Here is the relevant information that you need to answer an Alzheimer's patient's query.

BEGIN QUERY
{query}
END QUERY

BEGIN PATIENT PROFILE
{profile}
END PATIENT PROFILE

BEGIN INSTRUCTIONS
{instruction}
END INSTRUCTIONS
""".format(profile=profile, instruction=instruction, query=query)

In [ ]:
import time

responses = {}
scores = {}

for condition in system_prompts.keys():
  responses[condition] = []
  scores[condition] = []

for row in tq.tqdm(benchmark.iterrows(), total=len(benchmark)):
  for condition in system_prompts.keys():
    prompt = generate_prompt(row[1]['PatientProfile'], system_prompts[condition], row[1]['UserPrompt'])
    response = get_openai_response(model, prompt)
    responses[condition].append(response)
    scores[condition].append(distance_fn(response, row[1]['Ideal']))
    time.sleep(5)

  0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
import pandas as pd

# Create a dictionary to hold all data for the DataFrame
data_to_save = {}
for condition in system_prompts.keys():
    data_to_save[f'{condition}_response'] = responses[condition]
    data_to_save[f'{condition}_score'] = scores[condition]

# Create a DataFrame
results_df = pd.DataFrame(data_to_save)

# Save the DataFrame to a CSV file
results_df.to_csv('ablation_results.csv', index=False)

print("Responses and scores saved to 'ablation_results.csv'")

Responses and scores saved to 'ablation_results.csv'


In [ ]:
for condition in system_prompts.keys():
  print(f"Average {condition} scores: {np.mean(scores[condition])}")

Average full scores: 0.7462299455852234
Average ablation_priorities scores: 0.7454184886852758
Average ablation_directness scores: 0.7086799298516108
Average ablation_one_idea scores: 0.743934203060058
Average ablation_memory scores: 0.744913314951058
Average ablation_emotion scores: 0.7373730781278042
Average ablation_guidance scores: 0.7329484605858083
Average ablation_tone scores: 0.7437943455797127


In [ ]:
from scipy import stats
import itertools

# Get all keys from system_prompts
conditions = list(system_prompts.keys())

# Generate all unique pairs of conditions
pair_combinations = list(itertools.combinations(conditions, 2))

for condition1, condition2 in pair_combinations:
    if condition1 != "full" and condition2 != "full":
      continue
    print(f"\n--- Comparing {condition1} vs {condition2} ---")

    # Perform a paired t-test
    t_statistic, p_value = stats.ttest_rel(scores[condition1], scores[condition2])

    print(f"Paired t-test results:\n")
    print(f"T-statistic: {t_statistic:.4f}")
    print(f"P-value: {p_value}")

    if p_value < 0.05:
        print(f"\nThere is a statistically significant difference between the {condition1} and {condition2} scores.")
    else:
        print(f"\nThere is no statistically significant difference between the {condition1} and {condition2} scores.")

    # Calculate Cohen's d
    diffs = np.array(scores[condition2]) - np.array(scores[condition1])
    cohens_d = np.mean(diffs) / np.std(diffs)
    print(f"Cohen's d: {cohens_d:.4f}")
